In [1]:
from pathlib import Path
import urllib.request
import tarfile
import hashlib

data_dir = Path("/tmp/aclImdb")
archive = Path("/tmp/aclImdb_v1.tar.gz")
url = "https://huggingface.co/datasets/NolanChai/aclImdb_v1/resolve/main/aclImdb_v1.tar.gz"

if not archive.exists():
    partial = Path(str(archive) + ".part")
    urllib.request.urlretrieve(url, partial)
    partial.replace(archive)

checksum = hashlib.sha256(archive.read_bytes()).hexdigest()
assert checksum == "c40f74a18d3b61f90feba1e17730e0d38e8b97c05fde7008942e91923d1658fe"

if not data_dir.exists():
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall("/tmp", filter="data")

print("SHA256:", checksum)
print("Thư mục:", sorted(p.name for p in data_dir.iterdir()))

SHA256: c40f74a18d3b61f90feba1e17730e0d38e8b97c05fde7008942e91923d1658fe
Thư mục: ['README', 'imdb.vocab', 'imdbEr.txt', 'test', 'train']


In [2]:
import pandas as pd

rows = []

for split in ["train", "test"]:
    for category, label in [("neg", 0), ("pos", 1)]:
        for path in sorted((data_dir / split / category).glob("*.txt")):
            rows.append({
                "id": str(path.relative_to(data_dir)),
                "split": split,
                "text": path.read_text(encoding="utf-8"),
                "label": label
            })

df = pd.DataFrame(rows)

print("Số dòng, số cột:", df.shape)
display(df.head(10))
print("\nBài đánh giá đầu tiên:")
print(df.loc[0, "text"])

Số dòng, số cột: (50000, 4)


,id,split,text,label
0,train/neg/0_3.txt,train,Story of a man who has unnatural feelings for ...,0
1,train/neg/10000_4.txt,train,Airport '77 starts as a brand new luxury 747 p...,0
2,train/neg/10001_4.txt,train,This film lacked something I couldn't put my f...,0
3,train/neg/10002_1.txt,train,"Sorry everyone,,, I know this is supposed to b...",0
4,train/neg/10003_1.txt,train,When I was little my parents took me along to ...,0
5,train/neg/10004_3.txt,train,"""It appears that many critics find the idea of...",0
6,train/neg/10005_3.txt,train,The second attempt by a New York intellectual ...,0
7,train/neg/10006_4.txt,train,"I don't know who to blame, the timid writers o...",0
8,train/neg/10007_1.txt,train,This film is mediocre at best. Angie Harmon is...,0
9,train/neg/10008_2.txt,train,The film is bad. There is no other way to say ...,0



Bài đánh giá đầu tiên:
Story of a man who has unnatural feelings for a pig. Starts out with a opening scene that is a terrific example of absurd comedy. A formal orchestra audience is turned into an insane, violent mob by the crazy chantings of it's singers. Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting. Even those from the era should be turned off. The cryptic dialogue would make Shakespeare seem easy to a third grader. On a technical level it's better than you might think with some good cinematography by future great Vilmos Zsigmond. Future stars Sally Kirkland and Frederic Forrest can be seen briefly.


In [3]:
display(pd.crosstab(df["split"], df["label"]))

print("Giá trị thiếu:")
print(df.isna().sum())

print("Bài rỗng:", df["text"].str.strip().eq("").sum())
print("ID trùng:", df["id"].duplicated().sum())
print("Nội dung trùng:", df["text"].duplicated().sum())

label,0,1
split,,
test,12500,12500
train,12500,12500


Giá trị thiếu:
id       0
split    0
text     0
label    0
dtype: int64
Bài rỗng: 0
ID trùng: 0
Nội dung trùng: 418


In [4]:
print("Nội dung trùng trong từng tập:")
for split, part in df.groupby("split"):
    print(split, part["text"].duplicated().sum())

train_texts = set(df.loc[df["split"] == "train", "text"])
test_texts = set(df.loc[df["split"] == "test", "text"])

print("Nội dung xuất hiện ở cả train và test:",
      len(train_texts & test_texts))

duplicates = df[df["text"].duplicated(keep=False)]
print("Nội dung có nhiều nhãn:",
      (df.groupby("text")["label"].nunique() > 1).sum())

display(duplicates.sort_values("text").head(6))

print("Số từ mỗi bài:")
display(df["text"].str.split().str.len().describe())

Nội dung trùng trong từng tập:
test 199
train 96
Nội dung xuất hiện ở cả train và test: 123
Nội dung có nhiều nhãn: 0


,id,split,text,label
28814,test/neg/2183_2.txt,test,"""Go Fish"" garnered Rose Troche rightly or wron...",0
26084,test/neg/10977_2.txt,test,"""Go Fish"" garnered Rose Troche rightly or wron...",0
33153,test/neg/6089_3.txt,test,"""Three"" is a seriously dumb shipwreck movie. M...",0
30976,test/neg/4129_3.txt,test,"""Three"" is a seriously dumb shipwreck movie. M...",0
2552,train/neg/12298_4.txt,train,"""Witchery"" might just be the most incoherent a...",0
34165,test/neg/69_4.txt,test,"""Witchery"" might just be the most incoherent a...",0


Số từ mỗi bài:


count    50000.000000
mean       231.156940
std        171.343997
min          4.000000
25%        126.000000
50%        173.000000
75%        280.000000
max       2470.000000
Name: text, dtype: float64

In [6]:
import re
train = df[df["split"] == "train"]

patterns = {
    "Thẻ HTML": r"<[^>]+>",
    "Mã HTML": r"&(?:#\d+|#x[0-9a-fA-F]+|[a-zA-Z]+);",
    "Đường dẫn": r"https?://\S+|www\.\S+",
}

for name, pattern in patterns.items():
    mask = train["text"].str.contains(pattern, regex=True)
    print(f"\n{name}: {mask.sum()} bài")

    examples = train.loc[mask, "text"].head(2)
    for text in examples:
        match = re.search(pattern, text)
        start = max(0, match.start() - 60)
        end = min(len(text), match.end() + 60)
        print(repr(text[start:end]))


Thẻ HTML: 14667 bài
" survivor's as they await help with time fast running out...<br /><br />Also known under the slightly different tile Airport 1"
"alculated what he needed from the actors. I just don't know.<br /><br />But could it have been the screenplay? Just exactly wh"

Mã HTML: 7 bài
'If you are expecting to see a lot in the bath of Alt&#305;oklar, as it is promised by the tag line, you will be very d'
'b to really care.<br /><br />Eye Candy: Ashley Totin shows T&A; Evy Lutzky gets topless briefly; and Jennifer Hart shows he'

Đường dẫn: 95 bài
'e entertaining.<br /><br />- A review by Jamie Robert Ward (http://www.invocus.net)'
'ng out.<br /><br />** out of 4<br /><br />Roving Reviewer - www.geocities.com/paul_johnr'


In [7]:
import re

train = df[df["split"] == "train"].copy()

patterns = {
    "Thẻ HTML": r"<[^>]+>",
    "Mã HTML": r"&(?:#\d+|#x[0-9a-fA-F]+|[a-zA-Z]+);",
    "URL": r"https?://\S+|www\.\S+",
    "Email": r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b",
    "Khoảng trắng liên tiếp": r"\s{2,}",
    "Ký tự điều khiển": r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]",
    "Ký tự ngoài ASCII": r"[^\x00-\x7f]",
    "Số": r"\d",
    "Dấu câu lặp": r"[!?]{2,}",
    "Chữ viết hoa": r"\b[A-Z]{3,}\b",
    "Từ phủ định": r"\b(?:no|not|never|cannot)\b|\b\w+n['’]t\b",
}

stats = []

for name, pattern in patterns.items():
    mask = train["text"].str.contains(
        pattern, flags=re.IGNORECASE if name == "Từ phủ định" else 0,
        regex=True
    )
    stats.append({
        "Dấu hiệu": name,
        "Số bài": int(mask.sum()),
        "Tỷ lệ (%)": round(mask.mean() * 100, 2)
    })

display(pd.DataFrame(stats))

,Dấu hiệu,Số bài,Tỷ lệ (%)
0,Thẻ HTML,14667,58.67
1,Mã HTML,7,0.03
2,URL,95,0.38
3,Email,23,0.09
4,Khoảng trắng liên tiếp,216,0.86
5,Ký tự điều khiển,2,0.01
6,Ký tự ngoài ASCII,2347,9.39
7,Số,14164,56.66
8,Dấu câu lặp,2159,8.64
9,Chữ viết hoa,9445,37.78


In [8]:
for name, pattern in patterns.items():
    flags = re.IGNORECASE if name == "Từ phủ định" else 0
    mask = train["text"].str.contains(pattern, flags=flags, regex=True)

    print(f"\n{name}:")
    for text in train.loc[mask, "text"].head(2):
        match = re.search(pattern, text, flags=flags)
        start = max(0, match.start() - 70)
        end = min(len(text), match.end() + 70)
        print(repr(text[start:end]))

lengths = train["text"].str.split().str.len()

print("\n5 bài ngắn nhất:")
for idx in lengths.nsmallest(5).index:
    print(train.loc[idx, "id"], "|", lengths.loc[idx], "từ")
    print(train.loc[idx, "text"], "\n")

print("\n5 bài dài nhất:")
display(train.loc[lengths.nlargest(5).index, ["id", "label"]]
        .assign(word_count=lengths.nlargest(5)))


Thẻ HTML:
"nt for the survivor's as they await help with time fast running out...<br /><br />Also known under the slightly different tile Airport 1977 this s"
"r who miscalculated what he needed from the actors. I just don't know.<br /><br />But could it have been the screenplay? Just exactly who was the "

Mã HTML:
'If you are expecting to see a lot in the bath of Alt&#305;oklar, as it is promised by the tag line, you will be very disappointe'
'was to numb to really care.<br /><br />Eye Candy: Ashley Totin shows T&A; Evy Lutzky gets topless briefly; and Jennifer Hart shows her right ti'

URL:
'allow to be entertaining.<br /><br />- A review by Jamie Robert Ward (http://www.invocus.net)'
'rth checking out.<br /><br />** out of 4<br /><br />Roving Reviewer - www.geocities.com/paul_johnr'

Email:
' type to spoil a movie for anyone, but I INVITE anyone to email me at foxbarking@yahoo.com to ask for my opinion on this movie before they waste a dime on it. I'
"I agree with most if not all 

,id,label,word_count
14454,train/pos/1175_9.txt,1,2470
12549,train/pos/10044_9.txt,1,1839
18758,train/pos/4383_9.txt,1,1830
18417,train/pos/4076_10.txt,1,1723
16846,train/pos/2662_10.txt,1,1601


In [9]:
from collections import Counter
import unicodedata

train = df[df["split"] == "train"]

suspect_chars = Counter(
    char
    for text in train["text"]
    for char in text
    if unicodedata.category(char) == "Cc"
    and char not in "\n\r\t"
)

for char, count in suspect_chars.most_common():
    print(f"\nKý tự: {repr(char)} | U+{ord(char):04X} | Số lần: {count}")

    examples = train.loc[
        train["text"].str.contains(char, regex=False), "text"
    ].head(2)

    for text in examples:
        position = text.index(char)
        print(repr(text[max(0, position - 60):position + 61]))


Ký tự: '\x96' | U+0096 | Số lần: 1368
'nd three Superman-type villains, it started off with a bang \x96 the best Clark Kent to Superman transformations and rescue '
"e most hopelessly laughable 'horror' films I have ever seen \x96 in fact I still haven't recovered from the experience of wa"

Ký tự: '\x85' | U+0085 | Số lần: 707
' fact, they all but wrote the words: "Previously on Superman\x85" in S2RD. The special effects weren\'t great in either Part '
'cism, I will dignify the film now by explaining why it blows\x85 <br /><br />1. In a well-executed musical, the songs should'

Ký tự: '\x97' | U+0097 | Số lần: 283
'ing that wakes you up from your sleep while watching Tashan \x97 the mega-disappointing, mind-numbing new film at the cinema'
'e elements, a band of hostile Kiowa, an unscrupulous trader \x97 played by Donald Pleasance, here giving one of his most rid'

Ký tự: '\x91' | U+0091 | Số lần: 71
"ood fellas at Webster's Dictionary define Logophobia as the \x91fear of words'. I ma

In [10]:
import html
import re
import unicodedata

char_map = {}

for code in range(0x80, 0xA0):
    try:
        char_map[chr(code)] = bytes([code]).decode("cp1252")
    except UnicodeDecodeError:
        pass

def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)

    text = "".join(char_map.get(char, char) for char in text)
    text = unicodedata.normalize("NFC", text)

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b", " ", text)

    text = "".join(
        " " if unicodedata.category(char) == "Cc" else char
        for char in text
    )
    return re.sub(r"\s+", " ", text).strip()

df["clean_text"] = df["text"].apply(clean_text)

example_ids = train.loc[
    train["text"].str.contains(r"[\x85\x96\x8e]", regex=True)
].head(3).index

for idx in example_ids:
    print("\nFile:", df.loc[idx, "id"])
    print("Trước:", df.loc[idx, "text"])
    print("Sau:", df.loc[idx, "clean_text"])

print("\nBài rỗng sau xử lý:", df["clean_text"].eq("").sum())
print("Số bài thay đổi:", df["text"].ne(df["clean_text"]).sum())


File: train/neg/10055_3.txt
Trước: I grew up on the 'Superman II' theatrical version ("S2T") and as a kid, I loved it more than Part I since not only did it contain more Superman and three Superman-type villains, it started off with a bang  the best Clark Kent to Superman transformations and rescue scenes. Kids no longer had to impatiently wait for Superman to appear on screen, as in part I. Now as an adult, I can see how the mighty had fallen with S2T (See: my review.) I've always heard of the back-story on how they prematurely and unjustifiably fired the original's director, Richard Donner from part II. (It must have been a rarity back then to film two separate movies simultaneously, now it's common: 'Back to the Future' and 'Matrix' 2 & 3 for example.) Unfortunately, after finally seeing the Richard Donner Cut (or, "S2RD") I still can't fully recommend it. Gone, was the great Superman change scene, the entire Paris rescue, as was the wonderful recap of part I in S2T's opening. In 

In [11]:
checks = {
    "Bài rỗng": df["clean_text"].str.strip().eq(""),
    "Thẻ HTML": df["clean_text"].str.contains(r"<[^>]+>", regex=True),
    "URL": df["clean_text"].str.contains(r"https?://\S+|www\.\S+", regex=True),
    "Ký tự điều khiển": df["clean_text"].apply(
        lambda text: any(unicodedata.category(c) == "Cc" for c in text)
    ),
}

for name, mask in checks.items():
    print(f"{name}: {mask.sum()}")
    if mask.any():
        display(df.loc[mask, ["id", "clean_text"]].head(2))

print("\nNội dung trùng sau làm sạch:")
for split, part in df.groupby("split"):
    print(split, part["clean_text"].duplicated().sum())

train_texts = set(df.loc[df["split"] == "train", "clean_text"])
test_texts = set(df.loc[df["split"] == "test", "clean_text"])

print("Nội dung chung train/test:", len(train_texts & test_texts))
print(
    "Nội dung có nhiều nhãn:",
    (df.groupby("clean_text")["label"].nunique() > 1).sum()
)

Bài rỗng: 0
Thẻ HTML: 0
URL: 0
Ký tự điều khiển: 0

Nội dung trùng sau làm sạch:
test 201
train 98
Nội dung chung train/test: 123
Nội dung có nhiều nhãn: 0


In [12]:
from sklearn.model_selection import train_test_split

train_data = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

groups = train_data[["clean_text", "label"]].drop_duplicates("clean_text")

train_groups, dev_groups = train_test_split(
    groups,
    test_size=0.15,
    random_state=42,
    stratify=groups["label"]
)

is_dev = train_data["clean_text"].isin(dev_groups["clean_text"])

train_df = train_data.loc[~is_dev].copy()
dev_df = train_data.loc[is_dev].copy()

assert set(train_df["clean_text"]).isdisjoint(dev_df["clean_text"])
assert len(train_df) + len(dev_df) == 25000
assert len(test_df) == 25000

for name, part in [
    ("train", train_df),
    ("dev", dev_df),
    ("test", test_df)
]:
    print(f"\n{name}: {len(part)} bài")
    print(part["label"].value_counts().sort_index())

print("\nNội dung chung train/dev: 0")


train: 21247 bài
label
0    10623
1    10624
Name: count, dtype: int64

dev: 3753 bài
label
0    1877
1    1876
Name: count, dtype: int64

test: 25000 bài
label
0    12500
1    12500
Name: count, dtype: int64

Nội dung chung train/dev: 0


In [13]:
from pathlib import Path
import json

output_dir = Path("/kaggle/working/task1")
output_dir.mkdir(exist_ok=True)

for name, part in [
    ("train", train_df),
    ("dev", dev_df),
    ("test", test_df)
]:
    part.to_csv(output_dir / f"{name}.csv", index=False)

split_ids = {
    "train": train_df["id"].tolist(),
    "dev": dev_df["id"].tolist(),
    "test": test_df["id"].tolist()
}

with open(output_dir / "split_ids.json", "w", encoding="utf-8") as f:
    json.dump(split_ids, f, ensure_ascii=False, indent=2)

print("Đã lưu:", output_dir)
print("Các file:", sorted(p.name for p in output_dir.iterdir()))

Đã lưu: /kaggle/working/task1
Các file: ['dev.csv', 'split_ids.json', 'test.csv', 'train.csv']


In [14]:
from sklearn.feature_extraction.text import (
    TfidfVectorizer, ENGLISH_STOP_WORDS
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
import time

negations = {"no", "nor", "not", "never", "cannot"}

stopword_options = {
    "Không bỏ stopword": None,
    "Bỏ stopword": sorted(ENGLISH_STOP_WORDS),
    "Giữ phủ định": sorted(ENGLISH_STOP_WORDS - negations)
}

print("Từ phủ định được giữ lại:",
      sorted(ENGLISH_STOP_WORDS & negations))

Từ phủ định được giữ lại: ['cannot', 'never', 'no', 'nor', 'not']


In [15]:
stopword_results = []
stopword_models = {}

for name, stopwords in stopword_options.items():
    start = time.perf_counter()

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words=stopwords,
            token_pattern=r"(?u)\b\w+(?:['’]\w+)*\b",
            max_features=30000
        )),
        ("classifier", LogisticRegression(
            C=1.0,
            solver="liblinear",
            max_iter=1000,
            random_state=42
        ))
    ])

    model.fit(train_df["clean_text"], train_df["label"])
    predictions = model.predict(dev_df["clean_text"])

    stopword_results.append({
        "Cách xử lý": name,
        "Accuracy": accuracy_score(dev_df["label"], predictions),
        "F1": f1_score(dev_df["label"], predictions),
        "Số đặc trưng": len(model["tfidf"].vocabulary_),
        "Thời gian (giây)": round(time.perf_counter() - start, 2)
    })
    stopword_models[name] = model
    print("Đã chạy:", name)

stopword_results = pd.DataFrame(stopword_results)
display(stopword_results)

Đã chạy: Không bỏ stopword
Đã chạy: Bỏ stopword
Đã chạy: Giữ phủ định


,Cách xử lý,Accuracy,F1,Số đặc trưng,Thời gian (giây)
0,Không bỏ stopword,0.888356,0.889997,30000,6.71
1,Bỏ stopword,0.891021,0.892735,30000,5.62
2,Giữ phủ định,0.889955,0.891743,30000,5.62


In [16]:
from sklearn.feature_extraction.text import CountVectorizer

common = {
    "stop_words": stopword_options["Bỏ stopword"],
    "token_pattern": r"(?u)\b\w+(?:['’]\w+)*\b",
    "max_features": 30000
}

vectorizers = {
    "BoW unigram": CountVectorizer(**common),
    "TF-IDF unigram + bigram": TfidfVectorizer(
        **common, ngram_range=(1, 2)
    ),
    "TF-IDF ký tự": TfidfVectorizer(
        analyzer="char", ngram_range=(3, 5), max_features=30000
    )
}

baseline = stopword_results.iloc[1]

feature_results = [{
    "Biểu diễn": "TF-IDF unigram",
    "Accuracy": baseline["Accuracy"],
    "F1": baseline["F1"],
    "Thời gian (giây)": baseline["Thời gian (giây)"]
}]

feature_models = {
    "TF-IDF unigram": stopword_models["Bỏ stopword"]
}

for name, vectorizer in vectorizers.items():
    start = time.perf_counter()

    model = Pipeline([
        ("vectorizer", vectorizer),
        ("classifier", LogisticRegression(
            C=1.0, solver="liblinear",
            max_iter=1000, random_state=42
        ))
    ])

    model.fit(train_df["clean_text"], train_df["label"])
    predictions = model.predict(dev_df["clean_text"])

    feature_results.append({
        "Biểu diễn": name,
        "Accuracy": accuracy_score(dev_df["label"], predictions),
        "F1": f1_score(dev_df["label"], predictions),
        "Thời gian (giây)": round(time.perf_counter() - start, 2)
    })
    feature_models[name] = model
    print("Đã chạy:", name)

feature_results = pd.DataFrame(feature_results)
display(feature_results)

Đã chạy: BoW unigram
Đã chạy: TF-IDF unigram + bigram
Đã chạy: TF-IDF ký tự


,Biểu diễn,Accuracy,F1,Thời gian (giây)
0,TF-IDF unigram,0.891021,0.892735,5.62
1,BoW unigram,0.877431,0.879265,7.38
2,TF-IDF unigram + bigram,0.893952,0.895866,15.71
3,TF-IDF ký tự,0.883826,0.884595,92.72


In [17]:
%pip install -q gensim


Note: you may need to restart the kernel to use updated packages.


In [18]:
import numpy as np
from gensim.models import Word2Vec

start = time.perf_counter()

def tokenize(text):
    return re.findall(r"(?u)\b\w+(?:['’]\w+)*\b", text.lower())

train_tokens = train_df["clean_text"].apply(tokenize).tolist()
dev_tokens = dev_df["clean_text"].apply(tokenize).tolist()

w2v = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=5,
    sg=0,
    epochs=5,
    workers=1,
    seed=42
)

def average_vector(tokens, vectors):
    known = [word for word in tokens if word in vectors]
    if not known:
        return np.zeros(vectors.vector_size, dtype=np.float32)
    return vectors[known].mean(axis=0)

X_train_w2v = np.vstack([
    average_vector(tokens, w2v.wv) for tokens in train_tokens
])
X_dev_w2v = np.vstack([
    average_vector(tokens, w2v.wv) for tokens in dev_tokens
])

w2v_classifier = LogisticRegression(
    C=1.0, solver="liblinear", max_iter=1000, random_state=42
)
w2v_classifier.fit(X_train_w2v, train_df["label"])

predictions = w2v_classifier.predict(X_dev_w2v)

w2v_result = {
    "Biểu diễn": "Word2Vec CBOW trung bình",
    "Accuracy": accuracy_score(dev_df["label"], predictions),
    "F1": f1_score(dev_df["label"], predictions),
    "Thời gian (giây)": round(time.perf_counter() - start, 2)
}

print("Số từ trong Word2Vec:", len(w2v.wv))
print("Kích thước train:", X_train_w2v.shape)
display(pd.DataFrame([w2v_result]))

Số từ trong Word2Vec: 27831
Kích thước train: (21247, 100)


,Biểu diễn,Accuracy,F1,Thời gian (giây)
0,Word2Vec CBOW trung bình,0.805489,0.807996,67.17


In [19]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")

print("Số từ:", len(glove))
print("Số chiều:", glove.vector_size)

[==================================================] 100.0% 128.1/128.1MB downloaded
Số từ: 400000
Số chiều: 100


In [20]:
start = time.perf_counter()

X_train_glove = np.vstack([
    average_vector(tokens, glove) for tokens in train_tokens
])
X_dev_glove = np.vstack([
    average_vector(tokens, glove) for tokens in dev_tokens
])

glove_classifier = LogisticRegression(
    C=1.0, solver="liblinear", max_iter=1000, random_state=42
)
glove_classifier.fit(X_train_glove, train_df["label"])

predictions = glove_classifier.predict(X_dev_glove)

glove_result = {
    "Biểu diễn": "GloVe trung bình",
    "Accuracy": accuracy_score(dev_df["label"], predictions),
    "F1": f1_score(dev_df["label"], predictions),
    "Thời gian (giây)": round(time.perf_counter() - start, 2)
}

print("Kích thước train:", X_train_glove.shape)
print("Bài không có từ trong GloVe:",
      sum(not any(word in glove for word in tokens)
          for tokens in train_tokens))

display(pd.DataFrame([w2v_result, glove_result]))

Kích thước train: (21247, 100)
Bài không có từ trong GloVe: 0


,Biểu diễn,Accuracy,F1,Thời gian (giây)
0,Word2Vec CBOW trung bình,0.805489,0.807996,67.17
1,GloVe trung bình,0.807887,0.808703,16.14


In [21]:
%pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [22]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", device)

encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Giới hạn token:", encoder.max_seq_length)
start = time.perf_counter()

X_train_sentence = encoder.encode(
    train_df["clean_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_dev_sentence = encoder.encode(
    dev_df["clean_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

sentence_classifier = LogisticRegression(
    C=1.0, solver="liblinear", max_iter=1000, random_state=42
)
sentence_classifier.fit(X_train_sentence, train_df["label"])

predictions = sentence_classifier.predict(X_dev_sentence)

sentence_result = {
    "Biểu diễn": "SentenceTransformer MiniLM",
    "Accuracy": accuracy_score(dev_df["label"], predictions),
    "F1": f1_score(dev_df["label"], predictions),
    "Thời gian (giây)": round(time.perf_counter() - start, 2)
}

print("Kích thước train:", X_train_sentence.shape)
display(pd.DataFrame([sentence_result]))

Thiết bị: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (9 outputs left out)

Batches:   0%|          | 0/664 [00:00<?, ?it/s]

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Kích thước train: (21247, 384)


,Biểu diễn,Accuracy,F1,Thời gian (giây)
0,SentenceTransformer MiniLM,0.814548,0.814301,1492.12


In [23]:
from pathlib import Path
import json
import importlib.metadata as metadata

task2_dir = Path("/kaggle/working/task2")
task2_dir.mkdir(exist_ok=True)

all_feature_results = pd.concat([
    feature_results,
    pd.DataFrame([w2v_result, glove_result, sentence_result])
], ignore_index=True)

display(all_feature_results.sort_values("F1", ascending=False))

all_feature_results.to_csv(
    task2_dir / "feature_results.csv", index=False
)
stopword_results.to_csv(
    task2_dir / "stopword_results.csv", index=False
)

config = {
    "seed": 42,
    "train_size": len(train_df),
    "dev_size": len(dev_df),
    "metric": "binary F1, positive label = 1",
    "classifier": {
        "name": "LogisticRegression",
        "C": 1.0,
        "solver": "liblinear",
        "max_iter": 1000
    },
    "word_features": {
        "max_features": 30000,
        "stopwords": stopword_options["Bỏ stopword"],
        "token_pattern": common["token_pattern"]
    },
    "word2vec": {
        "method": "CBOW",
        "vector_size": 100,
        "window": 5,
        "min_count": 5,
        "epochs": 5,
        "workers": 1,
        "stopwords_removed": False,
        "pooling": "mean of known words"
    },
    "glove": "glove-wiki-gigaword-100, mean of known words",
    "sentence_transformer": {
        "model": "sentence-transformers/all-MiniLM-L6-v2",
        "max_seq_length": encoder.max_seq_length,
        "batch_size": 32,
        "device": device,
        "fine_tuned": False
    },
    "versions": {
        name: metadata.version(name)
        for name in [
            "scikit-learn", "gensim",
            "sentence-transformers", "torch", "numpy"
        ]
    }
}

(task2_dir / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Đã lưu:", task2_dir)

,Biểu diễn,Accuracy,F1,Thời gian (giây)
2,TF-IDF unigram + bigram,0.893952,0.895866,15.71
0,TF-IDF unigram,0.891021,0.892735,5.62
3,TF-IDF ký tự,0.883826,0.884595,92.72
1,BoW unigram,0.877431,0.879265,7.38
6,SentenceTransformer MiniLM,0.814548,0.814301,1492.12
5,GloVe trung bình,0.807887,0.808703,16.14
4,Word2Vec CBOW trung bình,0.805489,0.807996,67.17


Đã lưu: /kaggle/working/task2


In [24]:
import joblib

cache_dir = Path("/kaggle/working/artifacts/task2")
cache_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    cache_dir / "embeddings.npz",
    train_w2v=X_train_w2v,
    dev_w2v=X_dev_w2v,
    train_glove=X_train_glove,
    dev_glove=X_dev_glove,
    train_sentence=X_train_sentence,
    dev_sentence=X_dev_sentence
)

joblib.dump({
    "stopword_models": stopword_models,
    "feature_models": feature_models,
    "w2v_classifier": w2v_classifier,
    "glove_classifier": glove_classifier,
    "sentence_classifier": sentence_classifier
}, cache_dir / "models.joblib")

w2v.save(str(cache_dir / "word2vec.model"))

print("Đã lưu vector và mô hình:", cache_dir)

Đã lưu vector và mô hình: /kaggle/working/artifacts/task2


In [25]:
base_model = feature_models["TF-IDF unigram + bigram"]
vectorizer = base_model.named_steps["vectorizer"]

X_train_fs = vectorizer.transform(train_df["clean_text"])
X_dev_fs = vectorizer.transform(dev_df["clean_text"])

y_train = train_df["label"].to_numpy()
y_dev = dev_df["label"].to_numpy()
feature_names = vectorizer.get_feature_names_out()

print("Train:", X_train_fs.shape)
print("Dev:", X_dev_fs.shape)
print("10 đặc trưng đầu:", feature_names[:10])

Train: (21247, 30000)
Dev: (3753, 30000)
10 đặc trưng đầu: ['0' '0 10' '00' '000' '000 00' '000 000' '01' '02' '05' '06']


In [26]:
from sklearn.feature_selection import chi2, mutual_info_classif

X_presence = (X_train_fs > 0).astype(np.uint8).tocsc()

feature_scores = {}
selection_seconds = {}

start = time.perf_counter()
feature_scores["Tần suất"] = np.asarray(
    X_presence.sum(axis=0)
).ravel()
selection_seconds["Tần suất"] = time.perf_counter() - start

start = time.perf_counter()
feature_scores["Chi-square"] = chi2(X_presence, y_train)[0]
selection_seconds["Chi-square"] = time.perf_counter() - start

start = time.perf_counter()
feature_scores["Mutual information"] = mutual_info_classif(
    X_presence,
    y_train,
    discrete_features=True,
    random_state=42
)
selection_seconds["Mutual information"] = time.perf_counter() - start

for name, scores in feature_scores.items():
    top_indices = np.argsort(-scores, kind="stable")[:10]
    print(f"\n{name}:")
    print(feature_names[top_indices].tolist())
    print("Thời gian:", round(selection_seconds[name], 2), "giây")


Tần suất:
['movie', 'film', 'like', 'just', 'good', "it's", 'time', 'really', 'story', 'great']
Thời gian: 0.01 giây

Chi-square:
['worst', 'bad', 'waste', 'awful', 'great', 'excellent', 'stupid', 'waste time', 'worse', 'wonderful']
Thời gian: 0.14 giây

Mutual information:
['bad', 'worst', 'waste', 'awful', 'great', 'excellent', 'waste time', 'stupid', 'worse', 'wonderful']
Thời gian: 108.18 giây


In [27]:
fs_results = []
fs_indices = {}
fs_models = {}

def evaluate_features(name, indices, selection_time):
    start = time.perf_counter()

    model = LogisticRegression(
        C=1.0, solver="liblinear",
        max_iter=1000, random_state=42
    )
    model.fit(X_train_fs[:, indices], y_train)
    predictions = model.predict(X_dev_fs[:, indices])

    fs_results.append({
        "Cấu hình": name,
        "Số đặc trưng": len(indices),
        "Accuracy": accuracy_score(y_dev, predictions),
        "F1": f1_score(y_dev, predictions),
        "Chọn đặc trưng (giây)": round(selection_time, 2),
        "Train và dev (giây)": round(time.perf_counter() - start, 2)
    })

    fs_indices[name] = indices
    fs_models[name] = model
    print("Đã chạy:", name)

evaluate_features(
    "Không chọn lọc", np.arange(X_train_fs.shape[1]), 0
)

for method, scores in feature_scores.items():
    start = time.perf_counter()
    ranking = np.argsort(
        -np.nan_to_num(scores, nan=0.0), kind="stable"
    )
    ranking_time = time.perf_counter() - start

    for k in [5000, 15000]:
        evaluate_features(
            f"{method} - {k}",
            np.sort(ranking[:k]),
            selection_seconds[method] + ranking_time
        )

display(pd.DataFrame(fs_results))

Đã chạy: Không chọn lọc
Đã chạy: Tần suất - 5000
Đã chạy: Tần suất - 15000
Đã chạy: Chi-square - 5000
Đã chạy: Chi-square - 15000
Đã chạy: Mutual information - 5000
Đã chạy: Mutual information - 15000


,Cấu hình,Số đặc trưng,Accuracy,F1,Chọn đặc trưng (giây),Train và dev (giây)
0,Không chọn lọc,30000,0.893952,0.895866,0.00,0.48
1,Tần suất - 5000,5000,0.882494,0.884646,0.01,0.28
2,Tần suất - 15000,15000,0.888889,0.890866,0.01,0.39
3,Chi-square - 5000,5000,0.885958,0.888017,0.14,0.18
4,Chi-square - 15000,15000,0.891553,0.893539,0.14,0.30
5,Mutual information - 5000,5000,0.885691,0.887608,108.19,0.18
6,Mutual information - 15000,15000,0.891287,0.893250,108.19,0.30


In [28]:
for c in [0.1, 1.0]:
    start = time.perf_counter()

    selector = LogisticRegression(
        penalty="l1",
        C=c,
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )
    selector.fit(X_train_fs, y_train)

    indices = np.flatnonzero(
        np.abs(selector.coef_[0]) > 1e-5
    )
    selection_time = time.perf_counter() - start

    if len(indices) == 0:
        print(f"L1 C={c}: không chọn được đặc trưng")
        continue

    evaluate_features(
        f"L1 C={c}", indices, selection_time
    )

fs_results_df = pd.DataFrame(fs_results)
display(fs_results_df.sort_values("F1", ascending=False))

Đã chạy: L1 C=0.1
Đã chạy: L1 C=1.0


,Cấu hình,Số đặc trưng,Accuracy,F1,Chọn đặc trưng (giây),Train và dev (giây)
0,Không chọn lọc,30000,0.893952,0.895866,0.00,0.48
4,Chi-square - 15000,15000,0.891553,0.893539,0.14,0.30
6,Mutual information - 15000,15000,0.891287,0.893250,108.19,0.30
2,Tần suất - 15000,15000,0.888889,0.890866,0.01,0.39
3,Chi-square - 5000,5000,0.885958,0.888017,0.14,0.18
5,Mutual information - 5000,5000,0.885691,0.887608,108.19,0.18
1,Tần suất - 5000,5000,0.882494,0.884646,0.01,0.28
8,L1 C=1.0,834,0.874767,0.877156,0.50,0.11
7,L1 C=0.1,60,0.805222,0.813378,0.36,0.05


In [29]:
from pathlib import Path
import json
import joblib

task3_dir = Path("/kaggle/working/task3")
task3_dir.mkdir(exist_ok=True)

best_name = fs_results_df.loc[fs_results_df["F1"].idxmax(), "Cấu hình"]

fs_results_df.to_csv(
    task3_dir / "feature_selection_results.csv", index=False
)

config = {
    "representation": "TF-IDF unigram + bigram",
    "frequency": "number of train documents containing each feature",
    "chi_square_input": "binary presence",
    "mutual_information_input": "binary presence, discrete_features=True",
    "k_values": [5000, 15000],
    "l1_C_values": [0.1, 1.0],
    "l1_threshold": 1e-5,
    "classifier_C": 1.0,
    "classifier_solver": "liblinear",
    "classifier_max_iter": 1000,
    "seed": 42,
    "selection_metric": "dev binary F1, positive label = 1",
    "chosen_configuration": best_name,
    "chosen_feature_count": len(fs_indices[best_name]),
    "timing": "selection excludes TF-IDF creation; train/dev excludes selection"
}

(task3_dir / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

pd.DataFrame({
    "feature": feature_names[fs_indices[best_name]]
}).to_csv(task3_dir / "selected_features.csv", index=False)

artifact_dir = Path("/kaggle/working/artifacts/task3")
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "vectorizer": vectorizer,
    "indices": fs_indices[best_name],
    "classifier": fs_models[best_name]
}, artifact_dir / "selected_model.joblib")

print("Cấu hình chọn:", best_name)
print("Kết quả:", task3_dir)
print("Mô hình:", artifact_dir)

Cấu hình chọn: Không chọn lọc
Kết quả: /kaggle/working/task3
Mô hình: /kaggle/working/artifacts/task3


In [30]:
%pip install -q xgboost

Note: you may need to restart the kernel to use updated packages.


In [31]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import ParameterGrid
from xgboost import XGBClassifier
from pathlib import Path
import pandas as pd
import time
import json
import joblib

task4_dir = Path("/kaggle/working/task4")
model_dir = Path("/kaggle/working/artifacts/task4")
task4_dir.mkdir(exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

experiments = {
    "Naive Bayes": (
        MultinomialNB,
        {"alpha": [0.1, 1.0]},
        {}
    ),
    "Logistic Regression": (
        LogisticRegression,
        {"C": [0.1, 1.0, 10.0]},
        {"solver": "liblinear", "max_iter": 1000, "random_state": 42}
    ),
    "Linear SVM": (
        LinearSVC,
        {"C": [0.1, 1.0, 10.0]},
        {"dual": True, "max_iter": 5000, "random_state": 42}
    ),
    "XGBoost": (
        XGBClassifier,
        {"max_depth": [3, 6]},
        {
            "n_estimators": 200,
            "learning_rate": 0.1,
            "tree_method": "hist",
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "n_jobs": 4,
            "random_state": 42
        }
    )
}

ml_results = []
ml_best = {}

for name, (model_class, grid, fixed) in experiments.items():
    for params in ParameterGrid(grid):
        print(f"Đang chạy: {name} {params}", flush=True)
        model = model_class(**fixed, **params)

        start = time.perf_counter()
        model.fit(X_train_fs, y_train)
        train_seconds = time.perf_counter() - start

        start = time.perf_counter()
        predictions = model.predict(X_dev_fs)
        predict_seconds = time.perf_counter() - start

        result = {
            "Mô hình": name,
            "Cấu hình thử": json.dumps(params),
            "Accuracy": accuracy_score(y_dev, predictions),
            "F1": f1_score(y_dev, predictions),
            "Train (giây)": round(train_seconds, 2),
            "Dự đoán dev (giây)": round(predict_seconds, 2)
        }
        ml_results.append(result)

        if name not in ml_best or result["F1"] > ml_best[name]["F1"]:
            ml_best[name] = result.copy()

            joblib.dump({
                "vectorizer": vectorizer,
                "classifier": model
            }, model_dir / f"{name.replace(' ', '_')}.joblib")

        pd.DataFrame(ml_results).to_csv(
            task4_dir / "all_results.csv", index=False
        )
        print("F1 dev:", round(result["F1"], 4), flush=True)

ml_results_df = pd.DataFrame(ml_results)
ml_best_df = pd.DataFrame(ml_best.values())

ml_best_df.to_csv(task4_dir / "best_results.csv", index=False)
display(ml_best_df.sort_values("F1", ascending=False))

Đang chạy: Naive Bayes {'alpha': 0.1}
F1 dev: 0.8741
Đang chạy: Naive Bayes {'alpha': 1.0}
F1 dev: 0.8764
Đang chạy: Logistic Regression {'C': 0.1}
F1 dev: 0.87
Đang chạy: Logistic Regression {'C': 1.0}
F1 dev: 0.8959
Đang chạy: Logistic Regression {'C': 10.0}
F1 dev: 0.8984
Đang chạy: Linear SVM {'C': 0.1}
F1 dev: 0.8975
Đang chạy: Linear SVM {'C': 1.0}
F1 dev: 0.894
Đang chạy: Linear SVM {'C': 10.0}
F1 dev: 0.8797
Đang chạy: XGBoost {'max_depth': 3}
F1 dev: 0.8443
Đang chạy: XGBoost {'max_depth': 6}
F1 dev: 0.858


,Mô hình,Cấu hình thử,Accuracy,F1,Train (giây),Dự đoán dev (giây)
1,Logistic Regression,"{""C"": 10.0}",0.897149,0.898368,0.85,0.00
2,Linear SVM,"{""C"": 0.1}",0.895550,0.897490,0.20,0.00
0,Naive Bayes,"{""alpha"": 1.0}",0.875033,0.876416,0.02,0.00
3,XGBoost,"{""max_depth"": 6}",0.853717,0.857956,183.05,0.04


In [32]:
import numpy as np
import importlib.metadata as metadata

dev_predictions = []
chosen_configs = {}

for name, result in ml_best.items():
    filename = f"{name.replace(' ', '_')}.joblib"
    saved = joblib.load(model_dir / filename)

    X_check = saved["vectorizer"].transform(dev_df["clean_text"])
    predictions = saved["classifier"].predict(X_check)

    score = f1_score(y_dev, predictions)
    assert np.isclose(score, result["F1"], rtol=0, atol=1e-12)

    part = dev_df[["id", "label"]].copy()
    part["model"] = name
    part["prediction"] = predictions
    dev_predictions.append(part)

    chosen_configs[name] = {
        **experiments[name][2],
        **json.loads(result["Cấu hình thử"])
    }
    print("Nạp lại đúng:", name)

pd.concat(dev_predictions, ignore_index=True).to_csv(
    task4_dir / "dev_predictions.csv", index=False
)

best_model_name = ml_best_df.loc[
    ml_best_df["F1"].idxmax(), "Mô hình"
]

config = {
    "representation": "TF-IDF unigram + bigram, 30000 features",
    "train_size": len(train_df),
    "dev_size": len(dev_df),
    "selection_metric": "dev binary F1, positive label = 1",
    "best_model": best_model_name,
    "chosen_configs": chosen_configs,
    "search_grids": {
        name: specification[1]
        for name, specification in experiments.items()
    },
    "timing": "excludes TF-IDF creation; seconds rounded to 2 decimals",
    "versions": {
        name: metadata.version(name)
        for name in ["scikit-learn", "xgboost", "numpy", "joblib"]
    }
}

(task4_dir / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nMô hình chọn:", best_model_name)
print("File kết quả:", sorted(p.name for p in task4_dir.iterdir()))

Nạp lại đúng: Naive Bayes
Nạp lại đúng: Logistic Regression
Nạp lại đúng: Linear SVM
Nạp lại đúng: XGBoost

Mô hình chọn: Logistic Regression
File kết quả: ['all_results.csv', 'best_results.csv', 'config.json', 'dev_predictions.csv']


In [33]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

working = Path("/kaggle/working")
backup = working / "imdb_before_gpu.zip"

with ZipFile(backup, "w", compression=ZIP_DEFLATED) as archive:
    for folder in ["task1", "task2", "task3", "task4", "artifacts"]:
        for path in sorted((working / folder).rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(working))

print("Dung lượng MB:", round(backup.stat().st_size / 1024**2, 1))
display(FileLink(str(backup)))

Dung lượng MB: 105.9


/kaggle/working/imdb_before_gpu.zip

In [2]:
import torch
from pathlib import Path

print("PyTorch:", torch.__version__)
print("GPU khả dụng:", torch.cuda.is_available())

assert torch.cuda.is_available(), "Chưa nhận GPU"

device = torch.device("cuda:0")
print("GPU sử dụng:", torch.cuda.get_device_name(0))

data_path = Path("/kaggle/working/task1")
for name in ["train", "dev", "test"]:
    print(f"{name}.csv:", (data_path / f"{name}.csv").exists())

PyTorch: 2.11.0+cu130
GPU khả dụng: True
GPU sử dụng: Tesla T4
train.csv: False
dev.csv: False
test.csv: False


In [4]:
from pathlib import Path
import pandas as pd

files = [
    path for path in Path("/kaggle/input").rglob("train.csv")
    if (path.parent / "dev.csv").exists()
    and (path.parent / "test.csv").exists()
]

print("Đường dẫn tìm được:", files)
assert len(files) == 1, "Cần kiểm tra đường dẫn dữ liệu"

data_path = files[0].parent

train_df = pd.read_csv(data_path / "train.csv")
dev_df = pd.read_csv(data_path / "dev.csv")
test_df = pd.read_csv(data_path / "test.csv")

for name, part in [
    ("train", train_df),
    ("dev", dev_df),
    ("test", test_df)
]:
    print(name, part.shape)

Đường dẫn tìm được: [PosixPath('/kaggle/input/datasets/bangphan1311/imdb-backup/task1/train.csv')]
train (21247, 5)
dev (3753, 5)
test (25000, 5)


In [5]:
import re
import numpy as np
import random
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:0")

def tokenize_bilstm(text):
    return re.findall(r"(?u)\b\w+(?:['’]\w+)*\b", text.lower())

train_tokens = train_df["clean_text"].apply(tokenize_bilstm).tolist()
dev_tokens = dev_df["clean_text"].apply(tokenize_bilstm).tolist()

lengths = np.array([len(tokens) for tokens in train_tokens])

print("Độ dài train:")
for percentile in [50, 90, 95, 99]:
    print(f"P{percentile}:", np.percentile(lengths, percentile))

print("Bài không có token:", int((lengths == 0).sum()))
print("Ví dụ:", train_tokens[0][:20])

Độ dài train:
P50: 174.0
P90: 457.0
P95: 594.0
P99: 908.5400000000009
Bài không có token: 0
Ví dụ: ['story', 'of', 'a', 'man', 'who', 'has', 'unnatural', 'feelings', 'for', 'a', 'pig', 'starts', 'out', 'with', 'a', 'opening', 'scene', 'that', 'is', 'a']


In [6]:
from collections import Counter

MAX_VOCAB = 30000
MAX_LENGTH = 512
MIN_FREQ = 2

counts = Counter(
    word for tokens in train_tokens for word in tokens
)

vocab = {"<PAD>": 0, "<UNK>": 1}

for word, count in counts.most_common():
    if count >= MIN_FREQ and len(vocab) < MAX_VOCAB:
        vocab[word] = len(vocab)

def encode_tokens(tokens):
    ids = [vocab.get(word, 1) for word in tokens[:MAX_LENGTH]]
    return ids if ids else [1]

train_sequences = [encode_tokens(tokens) for tokens in train_tokens]
dev_sequences = [encode_tokens(tokens) for tokens in dev_tokens]

print("Kích thước từ điển:", len(vocab))
print("Bài train bị cắt:", sum(len(t) > MAX_LENGTH for t in train_tokens))
print("Bài dev bị cắt:", sum(len(t) > MAX_LENGTH for t in dev_tokens))
print("Token mẫu:", train_tokens[0][:15])
print("ID tương ứng:", train_sequences[0][:15])

Kích thước từ điển: 30000
Bài train bị cắt: 1588
Bài dev bị cắt: 307
Token mẫu: ['story', 'of', 'a', 'man', 'who', 'has', 'unnatural', 'feelings', 'for', 'a', 'pig', 'starts', 'out', 'with', 'a']
ID tương ứng: [61, 5, 4, 129, 34, 45, 7605, 1378, 15, 4, 4317, 510, 43, 16, 4]


In [7]:
%pip install -q gensim

Note: you may need to restart the kernel to use updated packages.


In [8]:
from pathlib import Path
from gensim.models import Word2Vec
import json

model_paths = list(
    Path("/kaggle/input").rglob("word2vec.model")
)

print("File Word2Vec:", model_paths)
assert len(model_paths) == 1, "Cần kiểm tra đường dẫn Word2Vec"

w2v = Word2Vec.load(str(model_paths[0]))
EMBEDDING_DIM = w2v.wv.vector_size

rng = np.random.default_rng(SEED)
embedding_std = float(w2v.wv.vectors.std())

embedding_matrix = rng.normal(
    0, embedding_std, (len(vocab), EMBEDDING_DIM)
).astype(np.float32)

matched = 0

for word, index in vocab.items():
    if index >= 2 and word in w2v.wv:
        embedding_matrix[index] = w2v.wv[word]
        matched += 1

embedding_matrix[0] = 0

task5_dir = Path("/kaggle/working/task5")
task5_dir.mkdir(exist_ok=True)

(task5_dir / "vocab.json").write_text(
    json.dumps(vocab, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Kích thước embedding:", embedding_matrix.shape)
print("Từ có vector Word2Vec:", matched)
print("Từ thường khởi tạo ngẫu nhiên:", len(vocab) - 2 - matched)
print("PAD là vector 0:", bool((embedding_matrix[0] == 0).all()))

File Word2Vec: [PosixPath('/kaggle/input/datasets/bangphan1311/imdb-backup/artifacts/task2/word2vec.model')]
Kích thước embedding: (30000, 100)
Từ có vector Word2Vec: 27831
Từ thường khởi tạo ngẫu nhiên: 2167
PAD là vector 0: True


In [9]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

config = {
    "seed": SEED,
    "vocab_size": len(vocab),
    "max_length": MAX_LENGTH,
    "min_frequency": MIN_FREQ,
    "embedding_dim": EMBEDDING_DIM,
    "embedding_source": "Word2Vec CBOW task2",
    "embedding_trainable": True,
    "unknown_initialization_std": embedding_std,
    "hidden_size": 128,
    "num_layers": 1,
    "bidirectional": True,
    "dropout": 0.3,
    "batch_size": 64,
    "epochs": 10,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "loss": "BCEWithLogitsLoss",
    "prediction_threshold": 0.5,
    "checkpoint_selection": "highest dev binary F1",
    "torch": str(torch.__version__),
    "gpu": torch.cuda.get_device_name(0)
}

def collate_batch(batch):
    sequences, labels = zip(*batch)
    sequences = [torch.tensor(s, dtype=torch.long) for s in sequences]
    lengths = torch.tensor([len(s) for s in sequences])
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded, lengths, torch.tensor(labels, dtype=torch.float32)

train_dataset = list(zip(train_sequences, train_df["label"].tolist()))
dev_dataset = list(zip(dev_sequences, dev_df["label"].tolist()))

generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    generator=generator,
    collate_fn=collate_batch
)
dev_loader = DataLoader(
    dev_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    collate_fn=collate_batch
)

class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_weights, hidden_size, dropout):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            embedding_weights, freeze=False, padding_idx=0
        )
        self.lstm = nn.LSTM(
            embedding_weights.shape[1],
            hidden_size,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, 1)

    def forward(self, tokens, lengths):
        embedded = self.embedding(tokens)
        packed = pack_padded_sequence(
            embedded, lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.lstm(packed)
        features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(self.dropout(features)).squeeze(1)

model = BiLSTMClassifier(
    torch.tensor(embedding_matrix),
    config["hidden_size"],
    config["dropout"]
).to(device)

config["parameters"] = sum(p.numel() for p in model.parameters())

print("Thiết bị:", next(model.parameters()).device)
print("Số tham số:", config["parameters"])
print("Batch train mỗi epoch:", len(train_loader))

Thiết bị: cuda:0
Số tham số: 3235777
Batch train mỗi epoch: 332


In [10]:
import time
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

artifact_dir = Path("/kaggle/working/artifacts/task5")
artifact_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = artifact_dir / "bilstm_best.pt"

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=config["learning_rate"]
)

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0
    gold, predicted = [], []

    with torch.set_grad_enabled(training):
        for tokens, lengths, labels in loader:
            tokens = tokens.to(device)
            targets = labels.to(device)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(tokens, lengths)
            loss = criterion(logits, targets)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(labels)
            gold.extend(labels.tolist())
            predicted.extend((logits.detach() >= 0).long().cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(gold, predicted),
        "f1": f1_score(gold, predicted)
    }

history = []
best_f1 = -1
best_epoch = None

torch.cuda.synchronize()
started = time.perf_counter()

for epoch in range(1, config["epochs"] + 1):
    epoch_start = time.perf_counter()

    train_metrics = run_epoch(train_loader, training=True)
    dev_metrics = run_epoch(dev_loader, training=False)

    torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - epoch_start

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_f1": train_metrics["f1"],
        "dev_loss": dev_metrics["loss"],
        "dev_accuracy": dev_metrics["accuracy"],
        "dev_f1": dev_metrics["f1"],
        "seconds": epoch_seconds
    })

    if dev_metrics["f1"] > best_f1:
        best_f1 = dev_metrics["f1"]
        best_epoch = epoch

        torch.save({
            "model_state_dict": model.state_dict(),
            "config": config,
            "vocab": vocab,
            "epoch": epoch,
            "dev_f1": best_f1
        }, checkpoint_path)

    pd.DataFrame(history).to_csv(
        task5_dir / "history.csv", index=False
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_metrics['loss']:.4f} | "
        f"dev loss {dev_metrics['loss']:.4f} | "
        f"dev F1 {dev_metrics['f1']:.4f} | "
        f"{epoch_seconds:.1f}s",
        flush=True
    )

config["training_with_dev_seconds"] = time.perf_counter() - started

(task5_dir / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nEpoch tốt nhất:", best_epoch)
print("F1 dev tốt nhất:", best_f1)
print("Checkpoint:", checkpoint_path)

display(pd.DataFrame(history))

Epoch 01 | train loss 0.6015 | dev loss 0.5068 | dev F1 0.7538 | 16.9s
Epoch 02 | train loss 0.4283 | dev loss 0.3702 | dev F1 0.8325 | 14.5s
Epoch 03 | train loss 0.2908 | dev loss 0.3017 | dev F1 0.8781 | 14.6s
Epoch 04 | train loss 0.1857 | dev loss 0.2890 | dev F1 0.8861 | 14.5s
Epoch 05 | train loss 0.1053 | dev loss 0.3221 | dev F1 0.8960 | 14.5s
Epoch 06 | train loss 0.0546 | dev loss 0.3930 | dev F1 0.8887 | 14.6s
Epoch 07 | train loss 0.0303 | dev loss 0.4702 | dev F1 0.8885 | 14.6s
Epoch 08 | train loss 0.0183 | dev loss 0.4876 | dev F1 0.8840 | 14.6s
Epoch 09 | train loss 0.0114 | dev loss 0.5409 | dev F1 0.8798 | 14.4s
Epoch 10 | train loss 0.0062 | dev loss 0.6341 | dev F1 0.8725 | 14.5s

Epoch tốt nhất: 5
F1 dev tốt nhất: 0.896
Checkpoint: /kaggle/working/artifacts/task5/bilstm_best.pt


,epoch,train_loss,train_f1,dev_loss,dev_accuracy,dev_f1,seconds
0,1,0.601548,0.665235,0.506774,0.756461,0.753772,16.921367
1,2,0.428301,0.801629,0.370210,0.845191,0.832517,14.526641
2,3,0.290768,0.878631,0.301651,0.875300,0.878125,14.565073
3,4,0.185708,0.931122,0.288990,0.890221,0.886125,14.482702
4,5,0.105304,0.964510,0.322092,0.896083,0.896000,14.477184
5,6,0.054556,0.984556,0.392981,0.885958,0.888658,14.588416
6,7,0.030278,0.991578,0.470184,0.886757,0.888539,14.607227
7,8,0.018336,0.995106,0.487641,0.882228,0.883990,14.633026
8,9,0.011400,0.996754,0.540878,0.879829,0.879829,14.420296
9,10,0.006229,0.998447,0.634097,0.867040,0.872542,14.480803


In [11]:
checkpoint = torch.load(
    checkpoint_path, map_location=device, weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

gold, predictions, probabilities = [], [], []

with torch.no_grad():
    for tokens, lengths, labels in dev_loader:
        logits = model(tokens.to(device), lengths)
        probs = torch.sigmoid(logits).cpu()

        gold.extend(labels.tolist())
        predictions.extend((probs >= 0.5).long().tolist())
        probabilities.extend(probs.tolist())

dev_f1 = f1_score(gold, predictions)
assert np.isclose(
    dev_f1, checkpoint["dev_f1"], rtol=0, atol=1e-12
)

result = {
    "model": "BiLSTM",
    "best_epoch": checkpoint["epoch"],
    "dev_accuracy": accuracy_score(gold, predictions),
    "dev_f1": dev_f1,
    "checkpoint_reload_verified": True,
    "training_with_dev_seconds": config["training_with_dev_seconds"],
    "parameters": config["parameters"],
    "gpu": config["gpu"]
}

(task5_dir / "result.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

prediction_df = dev_df[["id", "label"]].copy()
prediction_df["prediction"] = predictions
prediction_df["probability_positive"] = probabilities
prediction_df.to_csv(task5_dir / "dev_predictions.csv", index=False)

print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "model": "BiLSTM",
  "best_epoch": 5,
  "dev_accuracy": 0.8960831334932055,
  "dev_f1": 0.896,
  "checkpoint_reload_verified": true,
  "training_with_dev_seconds": 147.976076971,
  "parameters": 3235777,
  "gpu": "Tesla T4"
}


In [12]:
import shutil
from IPython.display import FileLink, display

shutil.make_archive(
    "/kaggle/working/task5_results",
    "zip",
    "/kaggle/working/task5"
)

display(FileLink("/kaggle/working/task5_results.zip"))

/kaggle/working/task5_results.zip

In [1]:
%pip install -q transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer

assert torch.cuda.is_available(), "Cần bật GPU trước"

device = torch.device("cuda:0")

data_path = Path(
    "/kaggle/input/datasets/bangphan1311/imdb-backup/task1"
)
train_df = pd.read_csv(data_path / "train.csv")
dev_df = pd.read_csv(data_path / "dev.csv")

assert len(train_df) == 21247
assert len(dev_df) == 3753
assert train_df["clean_text"].notna().all()
assert dev_df["clean_text"].notna().all()

transformer_config = {
    "seed": 42,
    "max_length": 256,
    "batch_size": 16,
    "gradient_accumulation_steps": 2,
    "epochs": 3,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,
    "precision": "fp16",
    "text_column": "clean_text",
    "checkpoint_selection": "highest dev binary F1",
    "torch": str(torch.__version__),
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0)
}

bert_name = "google-bert/bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)

bert_train_encodings = bert_tokenizer(
    train_df["clean_text"].tolist(),
    truncation=True,
    max_length=transformer_config["max_length"],
    padding=False
)
bert_dev_encodings = bert_tokenizer(
    dev_df["clean_text"].tolist(),
    truncation=True,
    max_length=transformer_config["max_length"],
    padding=False
)

print("GPU:", transformer_config["gpu"])
print("Transformers:", transformers.__version__)
print("Train:", len(bert_train_encodings["input_ids"]))
print("Dev:", len(bert_dev_encodings["input_ids"]))
print("Token mẫu:", bert_tokenizer.convert_ids_to_tokens(
    bert_train_encodings["input_ids"][0][:20]
))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

GPU: Tesla T4
Transformers: 5.20.2
Train: 21247
Dev: 3753
Token mẫu: ['[CLS]', 'story', 'of', 'a', 'man', 'who', 'has', 'unnatural', 'feelings', 'for', 'a', 'pig', '.', 'starts', 'out', 'with', 'a', 'opening', 'scene', 'that']


In [3]:
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
    set_seed
)
import math
import time
import json

set_seed(transformer_config["seed"])

class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: values[index]
            for key, values in self.encodings.items()
        }
        item["labels"] = int(self.labels[index])
        return item

bert_collator = DataCollatorWithPadding(
    tokenizer=bert_tokenizer,
    pad_to_multiple_of=8
)

bert_train_loader = DataLoader(
    ReviewDataset(bert_train_encodings, train_df["label"].tolist()),
    batch_size=transformer_config["batch_size"],
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
    collate_fn=bert_collator
)
bert_dev_loader = DataLoader(
    ReviewDataset(bert_dev_encodings, dev_df["label"].tolist()),
    batch_size=transformer_config["batch_size"],
    shuffle=False,
    collate_fn=bert_collator
)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_name,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1}
).to(device)

bert_result_dir = Path("/kaggle/working/task6/bert")
bert_checkpoint_dir = Path("/kaggle/working/artifacts/task6/bert")
bert_result_dir.mkdir(parents=True, exist_ok=True)
bert_checkpoint_dir.mkdir(parents=True, exist_ok=True)

bert_config = {
    **transformer_config,
    "model": bert_name,
    "source_revision": getattr(bert_model.config, "_commit_hash", None),
    "parameters": sum(p.numel() for p in bert_model.parameters())
}

(bert_result_dir / "config.json").write_text(
    json.dumps(bert_config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Thiết bị:", next(bert_model.parameters()).device)
print("Số tham số:", bert_config["parameters"])
print("Batch train mỗi epoch:", len(bert_train_loader))

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Thiết bị: cuda:0
Số tham số: 109483778
Batch train mỗi epoch: 1328


In [4]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

accumulation = transformer_config["gradient_accumulation_steps"]
total_steps = (
    math.ceil(len(bert_train_loader) / accumulation)
    * transformer_config["epochs"]
)

bert_optimizer = torch.optim.AdamW(
    bert_model.parameters(),
    lr=transformer_config["learning_rate"],
    weight_decay=transformer_config["weight_decay"]
)
bert_scheduler = get_linear_schedule_with_warmup(
    bert_optimizer,
    num_warmup_steps=int(total_steps * transformer_config["warmup_ratio"]),
    num_training_steps=total_steps
)
bert_scaler = torch.amp.GradScaler("cuda")

bert_history = []
bert_best_f1 = -1
bert_best_epoch = None

torch.cuda.synchronize()
bert_started = time.perf_counter()

for epoch in range(1, transformer_config["epochs"] + 1):
    epoch_start = time.perf_counter()
    bert_model.train()
    bert_optimizer.zero_grad(set_to_none=True)
    train_loss_sum = 0

    for step, batch in enumerate(tqdm(
        bert_train_loader, desc=f"BERT train {epoch}"
    )):
        batch = {key: value.to(device) for key, value in batch.items()}

        group_start = (step // accumulation) * accumulation
        group_size = min(
            accumulation, len(bert_train_loader) - group_start
        )

        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = bert_model(**batch)
            loss = outputs.loss

        train_loss_sum += loss.item() * batch["labels"].shape[0]
        bert_scaler.scale(loss / group_size).backward()

        if (step + 1) % accumulation == 0 or step + 1 == len(bert_train_loader):
            bert_scaler.unscale_(bert_optimizer)
            torch.nn.utils.clip_grad_norm_(
                bert_model.parameters(),
                transformer_config["max_grad_norm"]
            )

            previous_scale = bert_scaler.get_scale()
            bert_scaler.step(bert_optimizer)
            bert_scaler.update()

            if bert_scaler.get_scale() >= previous_scale:
                bert_scheduler.step()

            bert_optimizer.zero_grad(set_to_none=True)

    bert_model.eval()
    dev_loss_sum = 0
    gold, predictions = [], []

    with torch.no_grad():
        for batch in tqdm(bert_dev_loader, desc=f"BERT dev {epoch}"):
            batch = {key: value.to(device) for key, value in batch.items()}

            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = bert_model(**batch)

            dev_loss_sum += outputs.loss.item() * batch["labels"].shape[0]
            gold.extend(batch["labels"].cpu().tolist())
            predictions.extend(outputs.logits.argmax(dim=1).cpu().tolist())

    torch.cuda.synchronize()

    row = {
        "epoch": epoch,
        "train_loss": train_loss_sum / len(train_df),
        "dev_loss": dev_loss_sum / len(dev_df),
        "dev_accuracy": accuracy_score(gold, predictions),
        "dev_f1": f1_score(gold, predictions),
        "seconds": time.perf_counter() - epoch_start
    }
    bert_history.append(row)

    if row["dev_f1"] > bert_best_f1:
        bert_best_f1 = row["dev_f1"]
        bert_best_epoch = epoch

        bert_model.save_pretrained(bert_checkpoint_dir)
        bert_tokenizer.save_pretrained(bert_checkpoint_dir)

        prediction_df = dev_df[["id", "label"]].copy()
        prediction_df["prediction"] = predictions
        prediction_df.to_csv(
            bert_result_dir / "dev_predictions.csv", index=False
        )

    pd.DataFrame(bert_history).to_csv(
        bert_result_dir / "history.csv", index=False
    )

    print(
        f"Epoch {epoch} | train loss {row['train_loss']:.4f} | "
        f"dev loss {row['dev_loss']:.4f} | "
        f"dev F1 {row['dev_f1']:.4f} | {row['seconds']:.1f}s",
        flush=True
    )

bert_result = {
    "model": bert_name,
    "best_epoch": bert_best_epoch,
    "dev_f1": bert_best_f1,
    "training_with_dev_and_saving_seconds": time.perf_counter() - bert_started,
    "parameters": bert_config["parameters"],
    "gpu": transformer_config["gpu"]
}

(bert_result_dir / "result.json").write_text(
    json.dumps(bert_result, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

display(pd.DataFrame(bert_history))
print(json.dumps(bert_result, ensure_ascii=False, indent=2))

BERT train 1:   0%|          | 0/1328 [00:00<?, ?it/s]

BERT dev 1:   0%|          | 0/235 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1 | train loss 0.3267 | dev loss 0.2099 | dev F1 0.9160 | 305.6s


BERT train 2:   0%|          | 0/1328 [00:00<?, ?it/s]

BERT dev 2:   0%|          | 0/235 [00:00<?, ?it/s]

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 3 | train loss 0.0936 | dev loss 0.2887 | dev F1 0.9192 | 312.5s


,epoch,train_loss,dev_loss,dev_accuracy,dev_f1,seconds
0,1,0.326667,0.209909,0.915534,0.915982,305.572247
1,2,0.163805,0.232834,0.918732,0.918340,313.199445
2,3,0.093613,0.288745,0.918732,0.919205,312.486171


{
  "model": "google-bert/bert-base-uncased",
  "best_epoch": 3,
  "dev_f1": 0.919205298013245,
  "training_with_dev_and_saving_seconds": 933.3278955400001,
  "parameters": 109483778,
  "gpu": "Tesla T4"
}


In [5]:
bert_model.cpu()
torch.cuda.empty_cache()

bert_verified_model = AutoModelForSequenceClassification.from_pretrained(
    bert_checkpoint_dir
).to(device)
bert_verified_model.eval()

verified_predictions = []

with torch.no_grad():
    for batch in tqdm(bert_dev_loader, desc="Kiểm tra checkpoint BERT"):
        batch = {key: value.to(device) for key, value in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = bert_verified_model(**batch)

        verified_predictions.extend(
            outputs.logits.argmax(dim=1).cpu().tolist()
        )

verified_f1 = f1_score(dev_df["label"], verified_predictions)
saved_predictions = pd.read_csv(
    bert_result_dir / "dev_predictions.csv"
)

assert verified_predictions == saved_predictions["prediction"].tolist()
assert math.isclose(
    verified_f1, bert_result["dev_f1"], rel_tol=0, abs_tol=1e-12
)

bert_result["dev_accuracy"] = accuracy_score(
    dev_df["label"], verified_predictions
)
bert_result["checkpoint_reload_verified"] = True

(bert_result_dir / "result.json").write_text(
    json.dumps(bert_result, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(bert_result, ensure_ascii=False, indent=2))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Kiểm tra checkpoint BERT:   0%|          | 0/235 [00:00<?, ?it/s]

{
  "model": "google-bert/bert-base-uncased",
  "best_epoch": 3,
  "dev_f1": 0.919205298013245,
  "training_with_dev_and_saving_seconds": 933.3278955400001,
  "parameters": 109483778,
  "gpu": "Tesla T4",
  "dev_accuracy": 0.9187316813216094,
  "checkpoint_reload_verified": true
}


In [6]:
import gc

del bert_verified_model, bert_model
del bert_optimizer, bert_scheduler, bert_scaler
gc.collect()
torch.cuda.empty_cache()

set_seed(transformer_config["seed"])

roberta_name = "FacebookAI/roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)

roberta_train_encodings = roberta_tokenizer(
    train_df["clean_text"].tolist(),
    truncation=True,
    max_length=transformer_config["max_length"],
    padding=False
)
roberta_dev_encodings = roberta_tokenizer(
    dev_df["clean_text"].tolist(),
    truncation=True,
    max_length=transformer_config["max_length"],
    padding=False
)

roberta_collator = DataCollatorWithPadding(
    tokenizer=roberta_tokenizer,
    pad_to_multiple_of=8
)

roberta_train_loader = DataLoader(
    ReviewDataset(roberta_train_encodings, train_df["label"].tolist()),
    batch_size=transformer_config["batch_size"],
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
    collate_fn=roberta_collator
)
roberta_dev_loader = DataLoader(
    ReviewDataset(roberta_dev_encodings, dev_df["label"].tolist()),
    batch_size=transformer_config["batch_size"],
    shuffle=False,
    collate_fn=roberta_collator
)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_name,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1}
).to(device)

roberta_result_dir = Path("/kaggle/working/task6/roberta")
roberta_checkpoint_dir = Path("/kaggle/working/artifacts/task6/roberta")
roberta_result_dir.mkdir(parents=True, exist_ok=True)
roberta_checkpoint_dir.mkdir(parents=True, exist_ok=True)

roberta_config = {
    **transformer_config,
    "model": roberta_name,
    "source_revision": getattr(roberta_model.config, "_commit_hash", None),
    "parameters": sum(p.numel() for p in roberta_model.parameters())
}

(roberta_result_dir / "config.json").write_text(
    json.dumps(roberta_config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Thiết bị:", next(roberta_model.parameters()).device)
print("Số tham số:", roberta_config["parameters"])
print("Batch train mỗi epoch:", len(roberta_train_loader))

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Thiết bị: cuda:0
Số tham số: 124647170
Batch train mỗi epoch: 1328


In [7]:
accumulation = transformer_config["gradient_accumulation_steps"]
total_steps = (
    math.ceil(len(roberta_train_loader) / accumulation)
    * transformer_config["epochs"]
)

roberta_optimizer = torch.optim.AdamW(
    roberta_model.parameters(),
    lr=transformer_config["learning_rate"],
    weight_decay=transformer_config["weight_decay"]
)
roberta_scheduler = get_linear_schedule_with_warmup(
    roberta_optimizer,
    num_warmup_steps=int(total_steps * transformer_config["warmup_ratio"]),
    num_training_steps=total_steps
)
roberta_scaler = torch.amp.GradScaler("cuda")

roberta_history = []
roberta_best_f1 = -1
roberta_best_epoch = None

torch.cuda.synchronize()
roberta_started = time.perf_counter()

for epoch in range(1, transformer_config["epochs"] + 1):
    epoch_start = time.perf_counter()
    roberta_model.train()
    roberta_optimizer.zero_grad(set_to_none=True)
    train_loss_sum = 0

    for step, batch in enumerate(tqdm(
        roberta_train_loader, desc=f"RoBERTa train {epoch}"
    )):
        batch = {key: value.to(device) for key, value in batch.items()}

        group_start = (step // accumulation) * accumulation
        group_size = min(
            accumulation, len(roberta_train_loader) - group_start
        )

        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = roberta_model(**batch)
            loss = outputs.loss

        train_loss_sum += loss.item() * batch["labels"].shape[0]
        roberta_scaler.scale(loss / group_size).backward()

        if (step + 1) % accumulation == 0 or step + 1 == len(roberta_train_loader):
            roberta_scaler.unscale_(roberta_optimizer)
            torch.nn.utils.clip_grad_norm_(
                roberta_model.parameters(),
                transformer_config["max_grad_norm"]
            )

            previous_scale = roberta_scaler.get_scale()
            roberta_scaler.step(roberta_optimizer)
            roberta_scaler.update()

            if roberta_scaler.get_scale() >= previous_scale:
                roberta_scheduler.step()

            roberta_optimizer.zero_grad(set_to_none=True)

    roberta_model.eval()
    dev_loss_sum = 0
    gold, predictions = [], []

    with torch.no_grad():
        for batch in tqdm(roberta_dev_loader, desc=f"RoBERTa dev {epoch}"):
            batch = {key: value.to(device) for key, value in batch.items()}

            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = roberta_model(**batch)

            dev_loss_sum += outputs.loss.item() * batch["labels"].shape[0]
            gold.extend(batch["labels"].cpu().tolist())
            predictions.extend(outputs.logits.argmax(dim=1).cpu().tolist())

    torch.cuda.synchronize()

    row = {
        "epoch": epoch,
        "train_loss": train_loss_sum / len(train_df),
        "dev_loss": dev_loss_sum / len(dev_df),
        "dev_accuracy": accuracy_score(gold, predictions),
        "dev_f1": f1_score(gold, predictions),
        "seconds": time.perf_counter() - epoch_start
    }
    roberta_history.append(row)

    if row["dev_f1"] > roberta_best_f1:
        roberta_best_f1 = row["dev_f1"]
        roberta_best_epoch = epoch

        roberta_model.save_pretrained(roberta_checkpoint_dir)
        roberta_tokenizer.save_pretrained(roberta_checkpoint_dir)

        prediction_df = dev_df[["id", "label"]].copy()
        prediction_df["prediction"] = predictions
        prediction_df.to_csv(
            roberta_result_dir / "dev_predictions.csv", index=False
        )

    pd.DataFrame(roberta_history).to_csv(
        roberta_result_dir / "history.csv", index=False
    )

    print(
        f"Epoch {epoch} | train loss {row['train_loss']:.4f} | "
        f"dev loss {row['dev_loss']:.4f} | "
        f"dev F1 {row['dev_f1']:.4f} | {row['seconds']:.1f}s",
        flush=True
    )

roberta_result = {
    "model": roberta_name,
    "best_epoch": roberta_best_epoch,
    "dev_f1": roberta_best_f1,
    "training_with_dev_and_saving_seconds": time.perf_counter() - roberta_started,
    "parameters": roberta_config["parameters"],
    "gpu": transformer_config["gpu"]
}

(roberta_result_dir / "result.json").write_text(
    json.dumps(roberta_result, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

display(pd.DataFrame(roberta_history))
print(json.dumps(roberta_result, ensure_ascii=False, indent=2))

RoBERTa train 1:   0%|          | 0/1328 [00:00<?, ?it/s]

RoBERTa dev 1:   0%|          | 0/235 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1 | train loss 0.2969 | dev loss 0.2306 | dev F1 0.9206 | 316.9s


RoBERTa train 2:   0%|          | 0/1328 [00:00<?, ?it/s]

RoBERTa dev 2:   0%|          | 0/235 [00:00<?, ?it/s]

... (3 outputs left out)

RoBERTa dev 3:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch 3 | train loss 0.0911 | dev loss 0.2426 | dev F1 0.9344 | 312.4s


,epoch,train_loss,dev_loss,dev_accuracy,dev_f1,seconds
0,1,0.296905,0.230581,0.916600,0.920579,316.873199
1,2,0.150096,0.182613,0.935518,0.935225,313.579778
2,3,0.091124,0.242572,0.934719,0.934369,312.373963


{
  "model": "FacebookAI/roberta-base",
  "best_epoch": 2,
  "dev_f1": 0.9352248394004282,
  "training_with_dev_and_saving_seconds": 944.2575127060002,
  "parameters": 124647170,
  "gpu": "Tesla T4"
}


In [8]:
roberta_model.cpu()
torch.cuda.empty_cache()

roberta_verified_model = (
    AutoModelForSequenceClassification.from_pretrained(
        roberta_checkpoint_dir
    ).to(device)
)
roberta_verified_model.eval()

verified_predictions = []

with torch.no_grad():
    for batch in tqdm(roberta_dev_loader, desc="Kiểm tra checkpoint RoBERTa"):
        batch = {key: value.to(device) for key, value in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = roberta_verified_model(**batch)

        verified_predictions.extend(
            outputs.logits.argmax(dim=1).cpu().tolist()
        )

verified_f1 = f1_score(dev_df["label"], verified_predictions)
saved_predictions = pd.read_csv(
    roberta_result_dir / "dev_predictions.csv"
)

assert verified_predictions == saved_predictions["prediction"].tolist()
assert math.isclose(
    verified_f1, roberta_result["dev_f1"], rel_tol=0, abs_tol=1e-12
)

roberta_result["dev_accuracy"] = accuracy_score(
    dev_df["label"], verified_predictions
)
roberta_result["checkpoint_reload_verified"] = True

(roberta_result_dir / "result.json").write_text(
    json.dumps(roberta_result, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

task6_results = pd.DataFrame([bert_result, roberta_result])
task6_results.to_csv(
    "/kaggle/working/task6/comparison.csv", index=False
)

print(json.dumps(roberta_result, ensure_ascii=False, indent=2))
display(task6_results[[
    "model", "best_epoch", "dev_accuracy", "dev_f1",
    "training_with_dev_and_saving_seconds", "parameters"
]])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Kiểm tra checkpoint RoBERTa:   0%|          | 0/235 [00:00<?, ?it/s]

{
  "model": "FacebookAI/roberta-base",
  "best_epoch": 2,
  "dev_f1": 0.9352248394004282,
  "training_with_dev_and_saving_seconds": 944.2575127060002,
  "parameters": 124647170,
  "gpu": "Tesla T4",
  "dev_accuracy": 0.9355182520650147,
  "checkpoint_reload_verified": true
}


,model,best_epoch,dev_accuracy,dev_f1,training_with_dev_and_saving_seconds,parameters
0,google-bert/bert-base-uncased,3,0.918732,0.919205,933.327896,109483778
1,FacebookAI/roberta-base,2,0.935518,0.935225,944.257513,124647170


In [9]:
import shutil
from IPython.display import FileLink, display

shutil.make_archive(
    "/kaggle/working/task6_results",
    "zip",
    "/kaggle/working",
    "task6"
)

shutil.make_archive(
    "/kaggle/working/task6_models",
    "zip",
    "/kaggle/working/artifacts",
    "task6"
)

display(FileLink("/kaggle/working/task6_results.zip"))
display(FileLink("/kaggle/working/task6_models.zip"))

/kaggle/working/task6_results.zip

/kaggle/working/task6_models.zip